# Week 3: Monte Carlo Simulations

Craig Rudman  
Regis University  
MSDS684 Reinforcement Learning  
Prof. Mike Busch  

## Section 1: Project Overview 

This lab addresses Monte Carlo methods, a reinforcement learning approach that does not assume or require access to a model of environment dynamics in order to learn optimal policies for maximizing reward. The environment used for exploring this topic is Gymnasium's Blackjack-v1. The goal is to learn the optimal policy for hitting or sticking; that is requesting additional cards (hitting) or playing the hand with the cards already drawn (sticking).

The core concepts are presented in Sutton and Barto, Chapter 5.  In brief, the Monte Carlo methods learn optimal behavior from experience. This can be done directly with the environment or in simulation. This is an advantage over dynamic programming methods, but it comes at the cost of having to take a large number of samples.

Monte Carlo methods learn by averaging the returns on sample episodes. With sufficient exploration, enough samples are collected from each state to derive the average return. It is important that episodes from each start state are sampled; some mechanism must ensure sufficient state coverage for representative sampling. Also, some mechanism must ensure that all actions from each state are tried. When these conditions are met, the law of large numbers gives us confidence that the derived averages are sufficiently representative of the true mean. 

In the Blackjack example, we can assume that random dealing from an infinite deck over a large number of episodes will ensure that enough samples are collected from each start state. We will implement ε-greedy exploration with and without decay. With these approaches, we assume that enough state-action pairs will be evaluated; this is something we will verify by examining state-action visit counts.

We will be running the exercise using Gymnasium's Blackjack-v1 environment. The state space is defined as a tuple representing player sum in the range (4,21) with no usable ace, and (12,21) with a usable ace; the dealer's one shown card in the range (1,10); and whether or not the player has a usable ace (0,1). That means a total of 280 states are reachable in this environment: (18 * 10) + (10 * 10) = 280.

The action space has two options: hit (i.e. request another card), or stick (i.e. play with the sum already in hand). If the player sum goes over 21, then the episode ends with the player going bust. If the player chooses to stick, then the dealer hits according to policy until the dealer sticks or goes bust. In other words, episodes terminate when the player busts or when the dealer completes their turn, at which point the outcome is resolved.

The rewards in Blackjack are issued at episode termination: 1 for beating the dealer, 0 for a draw, and -1 for going bust or losing to the dealer. Thus, the return for the episode is the same as this reward. As a consequence, there is no discounting used in this simulation.

I expect that this simulation will yield a policy that is more conservative by sticking earlier when the dealer has cards between 2 and 6, but is more aggressive in hitting when the dealer is showing an ace, or cards 7 through 10. I also expect that we will find that 500,000 episodes provides sufficient coverage for episode starts and for state-action pairs.

## Section 2: Deliverables

GitHub Repository: https://github.com/craig-rudman/MSDS684/tree/main/W3

### Implementation Summary

This lab demonstrates first-visit Monte Carlo control with an ε-soft policy on Gymnasium's Blackjack-v1 environment, evaluating the effects of fixed epsilon values and decay schedules against naive and rules-based baselines.

The key components are: 

- a Blackjack Agent that selects actions and estimates Q-values by averaging returns across visits

- a Naive Random Agent and a Basic Strategy Agent serving as lower and upper performance baselines

- an Episode Runner that mediates between agent and environment

- five injectable decay schedules (linear, exponential, cosine, step, and adaptive) controlling how exploration gives way to exploitation over training

An initial run of 1,000,000 episodes establishes baselines. Three experiments, each using 1,000,000 episodes, are then conducted: 

- finding the optimal fixed epsilon (H1)

- identifying the best decay schedule (H2)

- testing whether adaptive decay outperforms fixed schedules (H3)

The NumPy random number generator is seeded with 42.

### Key Results and Analysis

#### Baseline Comparison

Three agents were compared over 1,000,000 training episodes:

- **Naive Random:** hits or sticks with equal probability
- **Basic Strategy:** rules-based play using hard and soft total lookup tables
- **MC Agent:** ε-soft policy with ε=0.01 and no decay

Table 1 shows the MC Agent substantially outperformed Naive Random (-0.060 vs -0.389) but fell slightly short of Basic Strategy (-0.046). The overlapping confidence intervals suggest this difference may not be practically meaningful.

| Agent | Avg Return | 95% CI | States | Avg Visits/State |
|---|---|---|---|---|
| Basic Strategy | -0.046 | [-0.054, -0.037] | | |
| MC Agent (ε=0.01, no decay)| -0.060 | [-0.068, -0.052] | 280 | 2488.3 |
| Naive Random | -0.389 | [-0.397, -0.382] | | |

*Table 1. Average return and 95% CI computed over the final 50,000 episodes.*

State coverage was complete: all 280 reachable states were visited, with a median of 2,946 visits per state (Table 2). However, coverage was uneven (std=6,199.7), reflecting the fact that high-sum states like 16-17 are encountered far more often than low-sum states.

| Metric | Value |
|---|---|
| States visited | 280 |
| Min visits per state | 425 |
| Max visits per state | 42,034 |
| Median visits per state | 2,946 |
| Std dev | 6,199.7 |

*Table 2. State visit statistics for the MC Agent after 1,000,000 training episodes (ε=0.01, no decay).*

Figure 1 confirms that the MC Agent improves steadily over training while both baselines remain flat.

<img src="../img/bakeoff_learning_curves.png" />

*Figure 1. Rolling average return over 1,000,000 episodes (window=50,000). Dashed lines indicate fixed baselines. Shading = 95% CI.*

After 1,000,000 episodes, the MC Agent has learned that high player totals with a useable ace win reliably, but mid-range sums lose consistently, especially against a strong dealer card. Players are at a disadvantage whenever dealers are showing 7 or better (Figure 2).

<img src="../img/value_surface_usable_ace.png">

*Figure 2. Estimated state-value function for hands with a usable ace, after 1,000,000 training episodes (ε=0.01, no decay)*

Comparing Figures 2 and 3, we can see that having a useable ace gives players a marginally better chance when player totals and dealer cards are in the midrange. Having a useable ace gives players a noticably higher expected return when the player total is 19 or higher.

<img src="../img/value_surface_no_usable_ace.png">

*Figure 3. Estimated state-value function for hands without a usable ace, after 1,000,000 training episodes (ε=0.01, no decay)*

#### Epsilon Experimentation

I trained agents with several different epsilon values, no decay. Table 3 shows that ε=0.05 achieved the best performance among MC agents, approaching but not quite matching Basic Strategy. Note that results varied slightly across runs; in one trial ε=0.01 performed best, while in another ε=0.05 edged ahead, suggesting the performance difference between these two values is within the margin of noise rather than a reliable distinction.

| Agent | Avg Return | 95% CI | States | Avg Visits/State |
|---|---|---|---|---|
| Basic Strategy | -0.046 | [-0.054, -0.037] | | |
| eps=0.05 | -0.058 | [-0.067, -0.050] | 280 | 2633.6 |
| eps=0.005 | -0.070 | [-0.078, -0.061] | 280 | 2419.6 |
| eps=0.01 | -0.074 | [-0.082, -0.065] | 280 | 2478.4 |
| eps=0.1 | -0.085 | [-0.093, -0.076] | 280 | 2661.2 |
| eps=0.001 | -0.102 | [-0.110, -0.094] | 280 | 2416.0 |
| eps=0.3 | -0.153 | [-0.161, -0.145] | 280 | 2565.8 |
| eps=0.5 | -0.215 | [-0.223, -0.206] | 280 | 2489.4 |
| eps=0.9 | -0.358 | [-0.366, -0.350] | 280 | 2430.0 |


*Table 3. Fixed epsilon experiment results after 1,000,000 training episodes. Agents sorted by average return descending. 95% CI computed over the final 50,000 episodes.*

The results suggest that too much exploration (i.e. high ε) floods the policy with random choices, preventing convergence. Too little exploration (i.e. low ε) causes the agent to exploit a poorly-sampled policy before it has learned enough. An optimal ε exists in the vicinity of ε=0.05 for this environment (Figure 4).

<img src="../img/exp1_epsilon_barchart.png">

*Figure 4. Average return by fixed epsilon value after 1,000,000 training episodes (tail=50,000). The red dashed line marks the BasicStrategy baseline (-0.040)*

#### Epsilon Decay

I tested four epsilon decay schedules, and one dynamically adaptive decay strategy. 

- Linear Decay: reduces ε by a fixed amount each episode until a minimum floor is reached

- Exponential Decay: multiplies ε by a constant factor each episode, slowing decay as ε approaches the floor

- Cosine Decay: anneals ε along a cosine curve from ε=1.0 to ε_min over a fixed number of steps

- Step Decay: drops ε by a fixed factor at regular episode intervals

- Adaptive Decay: adjusts the decay rate dynamically based on the agent's mean state-action visit count

The results in Table 4 show the results over 1,000,000 episodes.

| Agent | Avg Return | 95% CI | States | Avg Visits/State |
|---|---|---|---|---|
| Basic Strategy | -0.046 | [-0.054, -0.037] | | |
| Exponential | -0.049 | [-0.058, -0.041] | 280 | 2507.5 |
| Adaptive Decay | -0.061 | [-0.069, -0.053] | 280 | 2547.0 |
| Linear | -0.080 | [-0.089, -0.072] | 280 | 2410.1 |
| Cosine | -0.082 | [-0.091, -0.074] | 280 | 2378.0 |
| Step | -0.087 | [-0.095, -0.078] | 280 | 2380.3 |

*Table 4. Epsilon decay schedule results after 1,000,000 training episodes. Agents sorted by average return descending. 95% CI computed over the final 50,000 episodes. Adaptive Decay adjusts the decay rate dynamically based on mean state-action visit count.*

The Exponential Decay algorithm was the best strategy in this trial, almost beating the Basic Strategy agent. We should take note that the 95% confidence intervals of Basic Strategy, Exponential Decay, and Adaptive Decay overlap, and so the differences between them my not be meaningful.

When we examine the learning curves of all of the epsilon decay strategies, we can see that Exponential Decay is also the fastest to converge; by about 200,000 episodes, it has surpassed the Adaptive Decay strategy.

<img src="../img/all_decay_schedules.png" />

*Figure 5. Learning curve overlay for all epsilon decay schedules after 1,000,000 training episodes (window=50,000)*

#### Summary

Decay schedules consistently outperform fixed epsilon because they throttle exploration over time, shifting the balance toward exploitation as the policy matures. Exponential decay performed best in this trial, but run-to-run variability suggests the margin over other schedules is not reliable. Cross-validating results over multiple trials would be necessary to draw defensible conclusions about which schedule is optimal. The residual gap between the best MC agent and BasicStrategy likely reflects three factors: the epsilon floor permanently reserving a fraction of actions for exploration, the epsilon-soft policy occasionally taking suboptimal actions even with well-estimated Q-values, and the high variance of Blackjack outcomes; the overlapping confidence intervals suggest part of the gap may simply be noise.


## Section 3: AI Use Reflection

### 3.1 Initial Interaction

I asked Claude to summarize our initial interaction:

    Before writing any code, Craig and I defined the project architecture and development process together. We identified five classes with clear separation of concerns and established a test-driven iterative cycle: test first, develop, verify, accept. Craig directed that we sequence work by integration risk rather than class completion. All business logic lives in src/ as Python modules, with the notebook reserved for orchestration and experimentation.

### 3.2 Iteration Cycle

I asked Claude to describe our first four iterations:

    Across four cycles, Craig and I built the core infrastructure for a Monte Carlo Blackjack agent using strict test-first development. Cycle 1 established the foundation: BlackjackAgent for first-visit MC control and EpisodeRunner for environment interaction, with integration tests proving they work together. A design discussion led us to correct the action selection to a true epsilon-soft distribution for future decay support.

    Cycle 2 added verbose episode output and switched to Sutton and Barto mode. Cycle 3 introduced BasicStrategy, but Craig challenged my framing of it as a correctness benchmark. We agreed that policy agreement with uncited lookup tables isn't a meaningful success metric. Instead, we reframed evaluation around a performance bake-off comparing average returns across agents.

    Cycle 4 completed the bake-off trio with NaiveAgent. All three agents share a common interface compatible with EpisodeRunner, enabling direct comparison through episode returns rather than policy agreement. The project now has 43 passing tests, a clean separation of concerns, and a well-defined evaluation framework ready for training runs.

### 3.3 Critical Evaluation

Collaborating with an AI assistant introduces a subtle risk: each interaction is somewhat stateless, so the assistant may not carry forward awareness of prior design decisions, reuse opportunities, or experimental constraints. This can lead to redundant computation, inconsistent baselines, or poorly controlled experiments. I had to correct for several such examples. Visual inspection and automated unit tests were essential safeguards, alongside measuring state coverage, inspecting q-table values, and tracing execution of each method.

### 3.4 Learning Reflection

- The q-table maps each state to an array of action values, updated by rewinding the trajectory and averaging returns at the first visit of each state-action pair. 

- The epsilon-soft policy is elegant: each action gets probability ε/n, and the greedy action receives an additional 1−ε. 

- Comparing against a rules-based benchmark made optimization meaningful. 

- Results vary across runs; cross-validation would be necessary for rigorous conclusions.

## Section 4: Speaker Notes

Problem: Learn an optimal Blackjack policy from experience alone, without access to environment dynamics, using first-visit Monte Carlo control with an epsilon-soft policy.

Method: Agent plays 1,000,000 episodes, updating Q-values by rewinding each trajectory and averaging returns at first-visit state-action pairs. Policy is epsilon-soft: greedy action gets probability 1−ε+ε/n, all others get ε/n. Epsilon decay throttles exploration to improve later exploitation.

Design choice: Injected decay schedules as callables, keeping the agent decoupled from any specific exploration strategy. This made it easy to swap in linear, cosine, step, exponential, and adaptive decay without modifying the agent.

Key result: Fixed ε=0.05 performed best among MC agents (avg return −0.058), approaching but not matching BasicStrategy (−0.046). Exponential Decay slightly outperformed all fixed schedules.

Challenge: Controlling experiments fairly — episode count, eps_min, and baseline reuse had to be held constant across H1, H2, and H3 to make results comparable.

Insight: The epsilon sweet spot is narrow and asymmetric: too little exploration is more costly than slightly too much. The value surfaces confirm the agent learned meaningful structure — high player totals win, dealer strength dominates outcomes.

Connection: Monte Carlo requires complete episodes, which limits applicability. Computation burden grows exponentially as states become more complex. It requires a lot of samples to determine statistical validity.